# 🗣️ Conversation Evaluation Quickstart

This notebook shows the full loop for evaluating **multi-turn conversations** with TruLens:

1. Build a memory-enabled LangChain chatbot
2. Record 2–3 conversations, each tagged with a `conversation_id`
3. Run per-turn feedback metrics
4. Filter and compare results by conversation
5. Visualise in the TruLens dashboard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/quickstart/conversation_evaluation.ipynb)

In [ ]:
# !pip install trulens trulens-apps-langchain trulens-providers-openai langchain langchain-openai

In [ ]:
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "sk-proj-..."

## Build the chatbot

We use LangChain's `RunnableWithMessageHistory` to maintain per-conversation history.
Each conversation is keyed by a **session ID** — the same value we will pass as `conversation_id` to TruLens.

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a knowledgeable assistant. Answer concisely in 2–3 sentences.",
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# In-memory store keyed by session / conversation ID
store: dict = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


chatbot = RunnableWithMessageHistory(
    prompt | llm | StrOutputParser(),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

## Set up TruSession and feedback functions

In [ ]:
from trulens.core import TruSession

session = TruSession()
session.reset_database()

In [ ]:
from trulens.core import Metric
from trulens.core import Selector
from trulens.providers.openai import OpenAI

provider = OpenAI(model_engine="gpt-4o-mini")

# How well does the assistant's answer address the user's question?
f_answer_relevance = Metric(
    implementation=provider.relevance_with_cot_reasons,
    name="Answer Relevance",
    selectors={
        "prompt": Selector.select_record_input(),
        "response": Selector.select_record_output(),
    },
)

# Does the assistant's answer avoid introducing contradictions or hallucinations?
f_coherence = Metric(
    implementation=provider.coherence_with_cot_reasons,
    name="Coherence",
    selectors={
        "text": Selector.select_record_output(),
    },
)

## Wrap the chatbot with TruChain

In [ ]:
from trulens.apps.langchain import TruChain

tru_chatbot = TruChain(
    chatbot,
    app_name="Chatbot",
    app_version="v1",
    feedbacks=[f_answer_relevance, f_coherence],
)

## Run conversations

Each conversation gets a unique `conversation_id`.  
Every turn in the same conversation shares that ID so TruLens can group them together.

We run two independent conversations:
- **conv-climate-001** — a 3-turn discussion about climate change
- **conv-python-002** — a 3-turn discussion about Python programming

In [ ]:
# ── Conversation A: climate change ──────────────────────────────────────────
CONV_A = "conv-climate-001"

turns_a = [
    "What is the greenhouse effect?",
    "How does it relate to global warming?",
    "What are the most effective ways to reduce carbon emissions?",
]

for turn in turns_a:
    with tru_chatbot(conversation_id=CONV_A) as recording:
        chatbot.invoke(
            {"input": turn},
            config={"configurable": {"session_id": CONV_A}},
        )

In [ ]:
# ── Conversation B: Python programming ──────────────────────────────────────
CONV_B = "conv-python-002"

turns_b = [
    "What makes Python a good language for beginners?",
    "Can you explain list comprehensions with a simple example?",
    "When should I use a generator instead of a list?",
]

for turn in turns_b:
    with tru_chatbot(conversation_id=CONV_B) as recording:
        chatbot.invoke(
            {"input": turn},
            config={"configurable": {"session_id": CONV_B}},
        )

## View results

### Overall leaderboard

In [ ]:
session.get_leaderboard()

### Filter records by conversation

Use `conversation_id` to retrieve all turns for a single conversation and compare metrics turn-by-turn.

In [ ]:
records, feedback = session.get_records_and_feedback(app_ids=["Chatbot"])

# All turns for Conversation A
conv_a_records = records[records["conversation_id"] == CONV_A]
print(f"Conversation A — {len(conv_a_records)} turns")
conv_a_records[["input", "output", "Answer Relevance", "Coherence"]]

In [ ]:
# All turns for Conversation B
conv_b_records = records[records["conversation_id"] == CONV_B]
print(f"Conversation B — {len(conv_b_records)} turns")
conv_b_records[["input", "output", "Answer Relevance", "Coherence"]]

### Per-conversation aggregate scores

In [ ]:
summary = (
    records.groupby("conversation_id")[["Answer Relevance", "Coherence"]]
    .mean()
    .round(3)
)
summary

## Launch the TruLens dashboard

The dashboard's **Conversation** view lets you explore all turns of a conversation thread together,
inspect per-turn scores, and compare conversations side by side.

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)